In [1]:
import sys
import os
import json
import torch
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from transformers import logging
import ollama

# niech się tylko wyświetlają błędy
logging.set_verbosity_error()

# Model embeddingowy – zamienia tekst na wektory liczbowe (embeddingi)
# all-mpnet-base-v2 produkuje wektory o wymiarze 768
#embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
embedder = SentenceTransformer('BAAI/bge-m3')

# Model Ollama lokalnie pobrany przez ollama
ollama_model_name = "deepseek-r1:7b"
ollama_client = ollama.Client()

tokenizer = None
model = None

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Wilew\anaconda3\envs\rag_gpu\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Wilew\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [3]:
class Utils:
    def __init__(
        self,
        embedding_model: SentenceTransformer=None,
        llm_model=None,
        llm_tokenizer=None,
        ollama_client=None,
        ollama_model_name=None,
        index=None,
        metadata=None,
        chunk_size=512
    ):
        self.embedding_model = embedding_model
        self.llm_model = llm_model
        self.llm_tokenizer = llm_tokenizer
        self.ollama_client = ollama_client
        self.ollama_model_name = ollama_model_name
        self.index = index
        self.metadata = metadata
        # Rozmiar chunka w znakach – określa jak długie będą fragmenty tekstu
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        """
        Extract text from PDF file. Returns a list of tuples (page_number, text).
        """
        text = []
        pdf_document = pymupdf.open(pdf_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            # Zamieniamy znaki nowej linii na spacje, żeby tekst był ciągły
            text.append((page_num, str(page.get_text()).replace("\n", " ")))
        return text

    def chunk_text(self, text: list[tuple[int, str]]):
        chunks = []
        for page_num, page_text in text:
            # Dzielimy tekst strony na fragmenty o długości chunk_size znaków
            page_chunks = [
                (page_num, page_text[i:i+self.chunk_size])
                for i in range(0, len(page_text), self.chunk_size)
            ]
            chunks.extend(page_chunks)
        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc="Adding chunks to FAISS")):
            # Zamieniamy tekst chunka na wektor embeddingowy
            embeddings = self.embedding_model.encode(chunk, show_progress_bar=False)
            # Dodajemy wektor do indeksu FAISS
            self.index.add(np.array([embeddings]))
            # Zapisujemy metadane chunka – powiążemy je z wektorem przez pozycję w indeksie
            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })
        # Zapisujemy indeks FAISS na dysk, żeby nie trzeba było go odbudowywać przy każdym uruchomieniu
        faiss.write_index(self.index, db_loc + "vector_database.index")
        with open(db_loc + "metadata.json", "w") as file:
            json.dump(self.metadata, file)

    def process_file(self, file_path):
        """
        Process the file and add chunks to FAISS index
        """
        if file_path.endswith('.pdf'):
            text = self.extract_text_from_pdf(file_path)
        else:
            print(f"Unsupported file format, with extension: {os.path.splitext(file_path)[1]}")
            return 0

        chunks = self.chunk_text(text)
        self.add_chunks_to_faiss(chunks, filename=os.path.basename(file_path))
        return len(chunks)
    
    def answer_question(self, prompt_template="", query="", max_tokens=512, temp=0.7, k=5):
        # Zamieniamy pytanie użytkownika na wektor embeddingowy
        question_embedding = self.embedding_model.encode(query, show_progress_bar=False)

        # Szukamy k najbliższych wektorów w FAISS – D to odległości, I to indeksy znalezionych chunków
        D, I = self.index.search(np.array([question_embedding]), k)
        # Pobieramy metadane (tekst) znalezionych chunków
        chunks = [self.metadata[i] for i in I[0]]

        # Sklejamy teksty chunków w jeden blok kontekstu dla modelu
        context = ""
        for i, chunk in enumerate(chunks):
            context += f"{i+1}. {chunk['chunk']}\n"

        # Wstawiamy kontekst i pytanie do szablonu promptu
        prompt = prompt_template.format(context=context, query=query)

        # Budujemy historię rozmowy w formacie pasującym do Ollama
        messages = [
            {
                "role": "system",
                "content": (
                    "Be helpful, straight to the point. "
                    "Use only context. Do not hallucinate."
                )
            },
            {"role": "user", "content": prompt},
        ]

        # Generujemy odpowiedź lokalnie przez Ollama
        resp = self.ollama_client.chat(
            model=self.ollama_model_name,
            messages=messages,
            options={
                "temperature": temp,
                "num_predict": max_tokens
            }
        )

        answer = resp['message']['content']
        return answer, chunks

In [ ]:
knowledge_dir = "knowledge/"
db_dir = "vec_db/"
index_path = os.path.join(db_dir, "vector_database.index")
metadata_path = os.path.join(db_dir, "metadata.json")

# Sprawdzamy, czy pliki z bazą już istnieją na dysku
if os.path.exists(index_path) and os.path.exists(metadata_path):
    print("✅ Znaleziono gotową bazę wiedzy na dysku! Wczytywanie w ułamku sekundy...")
    
    # Odczytujemy zapisany wcześniej indeks FAISS
    index = faiss.read_index(index_path)
    
    # Odczytujemy zapisane teksty (metadane)
    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)
        
    print(f"Baza wczytana! Liczba fragmentów w pamięci: {index.ntotal}")

    # Tworzymy obiekt Utils korzystając z wczytanej przed chwilą bazy
    utils = Utils(
        embedding_model=embedder,
        llm_model=model,
        llm_tokenizer=tokenizer,
        ollama_client=ollama_client,
        ollama_model_name=ollama_model_name,
        index=index,
        metadata=metadata,
        chunk_size=1024*1024 
    )

else:
    print("⏳ Brak bazy na dysku. Rozpoczynam szatkowanie PDF-ów (to potrwa chwilę)...")
    
    # Tworzymy pusty indeks FAISS typu FlatL2 – przechowuje wektory i szuka po odległości euklidesowej
    # get_embedding_dimension() zwraca wymiar wektorów modelu embeddingowego (768)
    index = faiss.IndexFlatL2(embedder.get_embedding_dimension())

    # Lista słowników przechowująca metadane każdego chunka (nazwa pliku, numer strony, tekst)
    metadata = []
    
    # Tworzymy obiekt Utils z pustą bazą
    utils = Utils(
        embedding_model=embedder,
        llm_model=model,
        llm_tokenizer=tokenizer,
        ollama_client=ollama_client,
        ollama_model_name=ollama_model_name,
        index=index,
        metadata=metadata,
        chunk_size=1024*1024 
    )
    
    # Tworzymy folder na dysku, jeśli go nie ma
    if not os.path.exists(db_dir):
        os.makedirs(db_dir)

    # Przetwarzamy każdy plik PDF z katalogu – dzielimy na chunki i dodajemy do FAISS
    for file in os.listdir(knowledge_dir):
        utils.process_file(knowledge_dir + file)
        
    print('Number of chunks: ', index.ntotal)

Adding chunks to FAISS: 100%|██████████| 51/51 [01:55<00:00,  2.26s/it]


Number of chunks:  1851


In [6]:
# przukładowe pytania dotyczace powyzszych dokumentow
questions = [
    "Jakie są podstawowe wymiary płytki Nucleo-WBA55CG?",
    "Co posiada płytka Nucleo-WBA55CG?",
    "Jakie są parametry płytki Nucleo-WBA55CG?",
    "Jakie ma wymiary Nucleo-WBA55CG?",
    "Jak można zasilić Nucleo-WBA55CG?",
    "Ile pamięci posiada płytka Nucleo-WBA55CG?",
    "Do czego służy czarny przycisk w NUCLEO-C071RB?",
    "Jakie są wymiary NUCLEO-C071RB?",
    "Co posiada NUCLEO-C071RB?",
    "Jak zmienić zasilanie w NUCLEO-C071RB?",
    "Jakie cechy ma NUCLEO-C071RB?",
    "Jakie są charakterystyczne cechy płytki NUCLEO-C071RB w porównaniu do innych modeli?",
    "Czym się różni NUCLEO-WB05KZ od NUCLEO-WBA55CG?",
    "Jakimi cechami różni się NUCLEO-WB05KZ od NUCLEO-WBA55CG?",
    "Jakie parametry posiada płytka NUCLEO-WB05KZ, a jakie płytka NUCLEO-WBA55CG?",
]

In [10]:
from IPython.display import display, Markdown

# prompt_template ="""Based on the following context items, please answer the query.
# Give yourself room to think by extracting relevant passages from the context before answering the query.
# Don't return the thinking, only return the answer.
# Answer in Polish language only.
# Use the following examples as reference for the ideal answer style.
# Example 1:
# Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
# Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
# Now use the following context items to answer this one user query only:
# {context}
# Relevant passages: <extract relevant passages from the context here>
# Main User Query: {query}
# Answer:\n"""

prompt_template = """Jesteś ekspertem ds. systemów wbudowanych i mikrokontrolerów STM32. 
Twoim zadaniem jest odpowiadanie na pytania techniczne TYLKO I WYŁĄCZNIE na podstawie dostarczonego poniżej Kontekstu.

ZASADY:
1. Przeczytaj uważnie dostarczony Kontekst.
2. Odpowiedz na pytanie użytkownika opierając się TYLKO na informacjach z Kontekstu. 
3. Jeśli odpowiedź nie znajduje się w Kontekście, powiedz wprost: "Przepraszam, ale dostarczona dokumentacja nie zawiera informacji na ten temat." Nie zmyślaj i nie zgaduj (zero halucynacji).
4. Jeśli wymieniasz piny, rejestry lub nazwy płytek (np. NUCLEO), zachowaj ich oryginalną, dokładną pisownię.
5. Odpowiadaj w języku polskim, zwięźle i profesjonalnie. Używaj punktów, jeśli to ułatwi czytelność.

--- KONTEKST (fragmenty dokumentacji) ---
{context}

---
PYTANIE UŻYTKOWNIKA: {query}

ODPOWIEDŹ:
"""


# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=random_query,
        max_tokens=512,
        temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Jakie są podstawowe wymiary płytki Nucleo-WBA55CG?

**Odpowiedź:**



Podstawowe wymiary mechaniczne płytek Nucleo-WBA55CG to 76,0 x 53,0 x 18,0 mm.

---
**Źródła:**

**[1]** `um3301-stm32wba-nucleo64-board-mb1801-and-mb1803-stmicroelectronics.pdf` — strona 1

> Introduction NUCLEO-WBA55CG is a wireless and ultra-low-power board embedding a powerful and ultra‑low‑power radio compliant with  the Bluetooth® LE SIG specification v6.0, IEEE 802.15.4-2015 PHY and...

**[2]** `um3301-stm32wba-nucleo64-board-mb1801-and-mb1803-stmicroelectronics.pdf` — strona 8

> 7  Hardware layout and configuration NUCLEO-WBA55CG is designed around the STM32WBA55CG. The design includes a mezzanine board and an  MCU RF board. The hardware block diagram in Figure 2 illustrates...

**[3]** `um3301-stm32wba-nucleo64-board-mb1801-and-mb1803-stmicroelectronics.pdf` — strona 9

> Figure 3. NUCLEO-WBA55CG PCB top view DT59348V1 User push-buttons (B1, B2, and B3) User LEDs (LD1, LD2, and LD3) Reset push-button (B4) ST-LINK status LED (LD4) VDD jumper (JP2) 5V source selector (JP...

**[4]** `um3301-stm32wba-nucleo64-board-mb1801-and-mb1803-stmicroelectronics.pdf` — strona 39

> 8.2  NUCLEO-WBA55CG product history Table 18. Product history Order  code Product  identification Product details Product change description Product limitations NUCLEO-WBA55CG NUWBA55CG$DT1 MCU: STM32...

**[5]** `um3301-stm32wba-nucleo64-board-mb1801-and-mb1803-stmicroelectronics.pdf` — strona 10

> Figure 5. NUCLEO-WBA55CG PCB bottom view DT59350V2 SW1 (power switch) (ST-LINK USB Type-C® connector) CN15 Figure 6. NUCLEO-WBA55CG (MB1801D) mechanical dimensions (in millimeters) UM3301 Hardware lay...